# CineForge Kaggle Video Server — Wan T2V (photorealistic)

Port of the Colab server so you can run the **same** Wan open-weights video model on a free **Kaggle** GPU and expose it through the exact same HTTP API (so CineForge's existing `colab` backend works unchanged — just point `COLAB_BASE_URL` at the printed Kaggle URL).

* `GET  /health`   — status + which pipelines are loaded
* `POST /generate` — **text-to-video** (JSON body → MP4 bytes)
* `POST /image`    — **image-to-video** (multipart; only if a ≥30 GB GPU fits)

## ⚠️ STEP 0 — CONFIGURE KAGGLE FIRST, BEFORE RUNNING ANY CELL

Open the **Settings** panel (⚙️ icon, top-right).

| Setting | Value | Why |
|---------|-------|-----|
| **Accelerator** | **GPU P100** or **GPU T4** (16 GB) | Wan2.1 T2V 1.3B needs only ~8.2 GB VRAM |
| **Internet**    | **ON** (blue toggle) | installs packages + downloads model weights |

⚠️ **Internet OFF is the most common way these notebooks fail.** The install and model download need internet.

> Text-to-video needs ~8–16 GB of VRAM (we load in fp16). Image-to-video needs ≥30 GB, so on a free P100/T4 (16 GB) the notebook **skips** I2V with a clear warning — use the **kling**/**seedance** backends for image-to-video instead.

---

## How to use
1. Enable the Settings above, then **Save** (session will restart).
2. Click **▶ Run → "Restart & Run All"** and let it run through.
   Cell 1 installs the current Wan-AI-recommended stack (`pip install -U diffusers transformers accelerate`) plus the small extras Wan's pipeline imports, purges any stale package files left by Kaggle's image, then re-verifies by actually importing. Cell 7 prints a **PUBLIC URL**.
3. Paste it into `.env` as `COLAB_BASE_URL=<url>` (no trailing slash).
4. In the studio pick the **colab** backend (same HTTP API), or run:
   `python -m src.main gen --backend colab "your prompt"`

Keep this Kaggle tab open — like Colab, idle sessions disconnect and the URL dies with it (just re-run cell 7).

---

## Why this notebook exists (and why versions matter)
The old Colab notebook pinned `diffusers==0.31.0`, which **does not contain Wan at all** (`WanPipeline` first appears in diffusers **0.33.0**, and the image-to-video class is named `WanImageToVideoPipeline`, not `WanI2VPipeline`). Pinning old combos also caused `google.protobuf.runtime_version` import crashes from orphaned newer files in Kaggle's image. This notebook uses the vendor-recommended current stack instead.

In [ ]:
# ==========================================================================
#  CELL 1 - Install dependencies.
#
#  APPROACH CHANGE: no more ancient version pins. We install the CURRENT
#  Wan-AI-recommended stack and let diffusers + transformers co-resolve.  The
#  Wan-AI model card's canonical direction is:
#      pip install -U diffusers transformers accelerate
#      DiffusionPipeline.from_pretrained(<model>, dtype=..., device_map=...)
#
#  Wan2.1 T2V 1.3B needs only ~8.2 GB VRAM -> fits a 16 GB T4/P100 in fp16 with
#  NO bitsandbytes 8-bit, so we do not install bitsandbytes at all.
# ==========================================================================
import os
os.environ["CUDA_MODULE_LOADING"] = "EAGER"
import sys, importlib, subprocess, shutil
import torch

print("Python:", sys.version.split()[0])
print("torch  :", torch.__version__, "| CUDA build:", torch.version.cuda)

REQS = [
    "diffusers", "transformers", "accelerate",
    "sentencepiece", "protobuf", "einops",
    "imageio", "imageio-ffmpeg", "ftfy", "regex", "safetensors",
    "fastapi", "uvicorn", "nest-asyncio", "pydantic", "python-multipart",
    "psutil",
]

cmd = "pip install -q --upgrade " + " ".join(REQS)
print("Running:", cmd)
r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
print(r.stdout[-1500:] if len(r.stdout) > 1500 else r.stdout)
print(r.stderr[-1500:] if len(r.stderr) > 1500 else r.stderr)
print("pip exit:", r.returncode)

# Orphaned-file purge: Kaggle ships NEWER copies of some packages; a partial
# upgrade can leave stale dirs whose metadata lies. Delete the dirs then
# force-reinstall so on-disk code exactly matches the freshly-resolved versions.
HEAL_DIRS = ["google/protobuf", "transformers", "diffusers", "accelerate",
             "sentencepiece", "tokenizers", "safetensors", "einops",
             "imageio_ffmpeg"]

def heal_stack():
    tops = {d.split("/")[0] for d in HEAL_DIRS}
    for m in list(sys.modules):
        if m.split(".")[0] in tops:
            del sys.modules[m]
    nuked = 0
    for base in [p for p in sys.path if p and os.path.isdir(p)]:
        for rel in HEAL_DIRS:
            d = os.path.join(base, rel)
            if os.path.isdir(d):
                shutil.rmtree(d, ignore_errors=True)
                nuked += 1
    print("   purged stale dirs:", nuked)
    print("   re-installing resolved stack (2nd pass)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                    "--force-reinstall", "--no-deps"] + REQS,
                   capture_output=True, text=True)
    importlib.invalidate_caches()
    try:
        import google.protobuf as gp
        print("   protobuf:", getattr(gp, "__version__", "?"))
    except Exception as e:
        print("   protobuf import failed:", e)
        return False, nuked
    try:
        from diffusers import DiffusionPipeline, WanPipeline, \
            WanImageToVideoPipeline, AutoencoderKLWan
        from transformers import AutoTokenizer, UMT5EncoderModel
        print("   ✓ Wan + transformers imports OK")
        return True, nuked
    except Exception as e:
        print("   core import failed:", type(e).__name__, ":", str(e)[:300])
        return False, nuked

ok, nuked = heal_stack()

print("\n\n✅ install + repair finished.")
print("=" * 72)
if ok:
    print("🎉 Stack healthy (Wan-AI recommended versions).")
    print("   No manual restart needed. Continue cells 3 -> 8.")
else:
    print("🛑 core import still failed (see line above).")
    print("   Copy the 'core import failed:' line, then Session -> Restart,")
    print("   then Run All again.")
print("=" * 72)

---

## ℹ️ Cell 2 is just a note (no manual restart needed)

Cell 1 installs the **current** diffusers/transformers/accelerate (the combination Wan-AI actually tests against) and purges stale package files that Kaggle's image leaves behind, so you should **not** need to manually restart between cell 1 and cell 3.

* In the new Kaggle UI: **▶ Run → "Restart & Run All"**, let it finish.
* Cell 1 prints `🎉 Stack healthy ...` when the install + import check passed.
* If cell 1 still prints `🛑`, click **Session → Restart Session**, then **Run All** again.

Then cell 3 should be all ✅ and you're clear for cell 4 → 8.

---

In [ ]:
# ==========================================================================
#  CELL 3 - Environment sanity check.  If any line says FAIL, STOP and read.
#  NOTE: we do NOT pin exact diffusers/transformers versions here anymore -
#  we only require that the Wan classes actually import (that is what matters).
# ==========================================================================
import os, sys, importlib
os.environ["CUDA_MODULE_LOADING"] = "EAGER"
importlib.invalidate_caches()

# Safety net: re-run cell 1's repair before importing transformers/diffusers.
if "heal_stack" in globals():
    try:
        ok, nuked = heal_stack()
        print("stack heal re-applied (this session). dirs removed:", nuked)
    except Exception as e:
        print("stack heal re-apply raised:", e)
else:
    print("⚠️  heal_stack() not found - did you run cell 1 first? Run cell 1, then continue.")

_HAD_FAIL = [False]
def _ok(msg):   print(" ✅", msg)
def _fail(msg): print(" ❌ FAIL:", msg); _HAD_FAIL[0] = True

# 3a. packages present (no strict pins - latest is what Wan-AI recommends)
import importlib.metadata as md
for pkg in ["diffusers", "transformers", "accelerate", "sentencepiece",
            "safetensors", "einops", "ftfy", "regex", "imageio",
            "imageio_ffmpeg", "fastapi", "uvicorn", "pydantic"]:
    try:
        _ok(f"{pkg} {md.version(pkg)}")
    except md.PackageNotFoundError:
        _fail(f"{pkg} not installed. Re-run cell 1.")

# 3b. torch + CUDA + GPU
try:
    import torch
    _ok(f"torch {torch.__version__} (CUDA build {torch.version.cuda})")
except Exception as e:
    _fail(f"torch import: {e}"); torch = None

if torch is not None:
    if not torch.cuda.is_available():
        _fail("CUDA not visible. Settings -> Accelerator -> GPU P100/T4, then Save & re-run.")
    else:
        gpu = torch.cuda.get_device_name(0)
        vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
        _ok(f"GPU: {gpu}  ({vram_gb} GB VRAM)")
        if vram_gb < 14.0:
            print("   Warning: small GPU shape. Keep resolutions <=832x480, durations <=5 s.")
        _ = torch.zeros((1,), device="cuda")
        torch.cuda.synchronize()
        _ok("CUDA context warmed up.")

# 3c. RAM
try:
    import psutil
    ram_gb = round(psutil.virtual_memory().total / 1e9, 1)
    _ok(f"System RAM: {ram_gb} GB")
    if ram_gb < 10.0:
        print("   Warning: low-RAM shape.")
except Exception as e:
    _fail(f"psutil: {e}")

# 3d. critical imports (REAL imports - the same calls cell 4 makes).
try:
    import google.protobuf as gp
    _ok(f"google.protobuf {getattr(gp,'__version__','?')}")
except Exception as e:
    _fail(f"google.protobuf import: {e}")

try:
    from diffusers import DiffusionPipeline, WanPipeline, \
        WanImageToVideoPipeline, AutoencoderKLWan
    _ok("diffusers Wan OK: DiffusionPipeline, WanPipeline, WanImageToVideoPipeline, AutoencoderKLWan")
except Exception as e:
    _fail(f"diffusers Wan import: {type(e).__name__}: {e}")

try:
    from transformers import AutoTokenizer, UMT5EncoderModel
    _ok("transformers UMT5 OK (AutoTokenizer, UMT5EncoderModel)")
except Exception as e:
    _fail(f"transformers UMT5: {type(e).__name__}: {e}")

print()
if _HAD_FAIL[0]:
    raise SystemExit("\n🛑 Environment check FAILED. Fix errors above and re-run the cell.")
else:
    print("🎉 All environment checks PASSED. Safe to continue to cell 4.")

In [ ]:
# ==========================================================================
#  CELL 4 - Load the Wan pipelines onto the GPU.
#
#  Uses the CURRENT, Wan-AI-recommended loader: DiffusionPipeline.from_pretrained
#  with device_map - diffusers selects the exact pipeline (WanPipeline for T2V,
#  WanImageToVideoPipeline for I2V) and manages placement. No bitsandbytes, no
#  hand-assembly of tokenizer/text_encoder/vae.
#
#  * T2V (text-to-video): always loaded
#  * I2V (image-to-video): loaded ONLY if VRAM >= 30 GB (not on P100/T4)
# ==========================================================================
import gc, os, sys
os.environ["CUDA_MODULE_LOADING"] = "EAGER"

# Self-heal core packages BEFORE importing transformers/diffusers.
if "heal_stack" in globals():
    try:
        heal_stack()
    except Exception as e:
        print("stack heal at cell 4 raised:", e)

import torch, psutil
import transformers, diffusers
from diffusers import DiffusionPipeline
from transformers import AutoTokenizer, UMT5EncoderModel

# Keep HF weights cache on Kaggle's big writable disk (12 GB quota).
os.environ["HF_HOME"] = "/root/.cache/huggingface"

T2V_MODEL_ID = os.getenv("T2V_MODEL_ID", "Wan-AI/Wan2.1-T2V-1.3B-Diffusers")
I2V_MODEL_ID = os.getenv("I2V_MODEL_ID", "Wan-AI/Wan2.1-I2V-1.3B-Diffusers")

if not torch.cuda.is_available():
    raise SystemExit("No CUDA GPU - check Settings -> Accelerator = GPU.")

VRAM_GB = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
LOAD_I2V = VRAM_GB >= 30
print(f"GPU: {torch.cuda.get_device_name(0)}  ({VRAM_GB} GB)")
print(f"I2V pipeline will be loaded: {LOAD_I2V}")
print(f"diffusers {diffusers.__version__} | transformers {transformers.__version__}")

# T4/P100 do not benefit from bf16; use fp16 everywhere.
DT = torch.float16

print("④ Loading Wan T2V pipeline - this is the big download+load step ...")
try:
    pipe = DiffusionPipeline.from_pretrained(
        T2V_MODEL_ID, torch_dtype=DT, device_map="cuda",
    )
    _offload = False
except Exception as e:
    print("   device_map load failed:", type(e).__name__, str(e)[:160])
    print("   Falling back to .to(cuda) + enable_model_cpu_offload() ...")
    pipe = DiffusionPipeline.from_pretrained(T2V_MODEL_ID, torch_dtype=DT)
    try:
        pipe.enable_vae_tiling()
    except AttributeError:
        pass
    pipe.enable_model_cpu_offload()
    _offload = True

# Make sure it is exactly the Wan T2V pipeline type we expect.
if type(pipe).__name__ != "WanPipeline":
    print("WARNING: model did not auto-select WanPipeline (got", type(pipe).__name__, ")")

# ---- I2V: Kaggle free GPUs are 16 GB, so it is skipped by default ----------
i2v_pipe = None
if LOAD_I2V:
    print("⑤ Loading Wan I2V pipeline (image-to-video) ...")
    try:
        i2v_pipe = DiffusionPipeline.from_pretrained(I2V_MODEL_ID, torch_dtype=DT, device_map="cuda")
    except Exception:
        i2v_pipe = DiffusionPipeline.from_pretrained(I2V_MODEL_ID, torch_dtype=DT)
        i2v_pipe.enable_model_cpu_offload()
else:
    print("⑤ Skipping I2V pipeline (<30 GB VRAM). The /image endpoint returns 501;")
    print("   use the kling/seedance backends for image-to-video.")

print()
print("=" * 72)
print(f"✅ Wan ready on {torch.cuda.get_device_name(0)}")
print("   Pipeline type:", type(pipe).__name__)
print("   T2V model :", T2V_MODEL_ID)
print("   I2V model :", I2V_MODEL_ID if i2v_pipe is not None else "(not loaded)")
print("   VRAM used :", round(torch.cuda.memory_allocated() / 1e9, 1), "/", VRAM_GB, "GB")
print("   cpu_offload:", _offload)

In [ ]:
# ==========================================================================
#  CELL 5 — FastAPI HTTP server the CineForge colab backend calls.
#
#    GET  /health    -> status + which pipelines are loaded
#    POST /generate  -> text-to-video   (JSON body, returns MP4)
#    POST /image     -> image-to-video  (multipart; only if I2V was loaded)
#
#  This is API-compatible with the Colab notebook, so CineForge's existing
#  'colab' backend works against this server with zero code changes.
# ==========================================================================
import io as _io
import os, random, tempfile, threading, time
from typing import Optional

import torch
from PIL import Image
from diffusers.utils import export_to_video
from fastapi import FastAPI, File, Form, UploadFile
from fastapi.responses import JSONResponse, Response
from pydantic import BaseModel, Field

app = FastAPI(title="CineForge Kaggle Video Server (Wan T2V)")
GEN_LOCK = threading.Lock()

DEFAULT_NEGATIVE = (
    "cartoon, anime, illustration, painting, drawing, CGI, 3d render, "
    "plastic skin, blurry, low quality, worst quality, jpeg artifacts, deformed, "
    "extra limbs, missing fingers, bad hands, bad teeth"
)

def _snap(v, lo=64, multiple=16):
    return max(lo, (int(v) // multiple) * multiple)

def _frames_to_response(frames, fps, seed, elapsed, w, h, model_name):
    with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as f:
        tmp_path = f.name
    try:
        export_to_video(frames, tmp_path, fps=fps)
        data = open(tmp_path, "rb").read()
    finally:
        try:
            os.unlink(tmp_path)
        except OSError:
            pass
    return Response(
        content=data, media_type="video/mp4",
        headers={
            "X-Seed": str(seed), "X-Fps": str(fps), "X-Model": model_name,
            "X-Elapsed-S": str(elapsed), "X-Width": str(w), "X-Height": str(h),
        },
    )

class GenBody(BaseModel):
    prompt: str
    negative_prompt: str = ""
    width: int = Field(default=832, ge=64, le=1280)
    height: int = Field(default=480, ge=64, le=1280)
    fps: int = Field(default=16, ge=5, le=30)
    duration: float = Field(default=5.0, ge=1.0, le=10.0)
    num_inference_steps: int = Field(default=25, ge=4, le=50)
    guidance_scale: float = Field(default=5.0, ge=0.0, le=15.0)
    seed: Optional[int] = None

@app.get("/health")
def health():
    return {
        "status": "ok", "t2v_model": T2V_MODEL_ID,
        "i2v_model": I2V_MODEL_ID if i2v_pipe is not None else None,
        "i2v_available": i2v_pipe is not None, "busy": GEN_LOCK.locked(),
        "vram_gb": round(torch.cuda.memory_allocated() / 1e9, 2),
    }

@app.post("/generate")
def generate(body: GenBody):
    negative = body.negative_prompt or DEFAULT_NEGATIVE
    seed = body.seed if body.seed is not None else random.randint(0, 2**31 - 1)
    gen = torch.Generator("cuda").manual_seed(seed)
    num_frames = max(9, min(int(body.duration * body.fps), 121))
    w, h = _snap(body.width), _snap(body.height)
    t0 = time.time()
    with GEN_LOCK:
        try:
            out = pipe(
                prompt=body.prompt, negative_prompt=negative,
                width=w, height=h, num_frames=num_frames,
                num_inference_steps=body.num_inference_steps,
                guidance_scale=body.guidance_scale, generator=gen,
                max_sequence_length=512,
            ).frames[0]
        except torch.cuda.OutOfMemoryError as e:
            gc.collect(); torch.cuda.empty_cache()
            return JSONResponse(status_code=507, content={
                "error": "CUDA OOM", "detail": str(e),
                "hint": "Lower resolution (e.g. 768x432) or shorter duration (<4 s)."})
    return _frames_to_response(out, body.fps, seed, round(time.time() - t0, 1), w, h, T2V_MODEL_ID)

@app.post("/image")
async def image_to_video(
    file: UploadFile = File(...),
    prompt: str = Form(""),
    negative_prompt: str = Form(""),
    width: int = Form(832, ge=64, le=1280),
    height: int = Form(480, ge=64, le=1280),
    fps: int = Form(16, ge=5, le=30),
    duration: float = Form(5.0, ge=1.0, le=10.0),
    num_inference_steps: int = Form(25, ge=4, le=50),
    guidance_scale: float = Form(5.0, ge=0.0, le=15.0),
    seed: Optional[int] = Form(None),
):
    if i2v_pipe is None:
        return JSONResponse(status_code=501, content={
            "error": "I2V pipeline not loaded",
            "hint": "Image-to-video needs a >=30 GB GPU (A100). On a free Kaggle P100/T4 "
                    "use the kling/seedance backends for I2V."})
    raw = await file.read()
    try:
        img = Image.open(_io.BytesIO(raw)).convert("RGB")
    except Exception as e:
        return JSONResponse(status_code=400, content={"error": f"could not decode image: {e}"})

    negative = negative_prompt or DEFAULT_NEGATIVE
    final_seed = seed if seed is not None else random.randint(0, 2**31 - 1)
    gen = torch.Generator("cuda").manual_seed(final_seed)
    num_frames = max(9, min(int(duration * fps), 121))
    w, h = _snap(width), _snap(height)
    t0 = time.time()
    with GEN_LOCK:
        try:
            out = i2v_pipe(
                prompt=prompt or ("cinematic motion, natural camera movement"),
                negative_prompt=negative, image=img,
                width=w, height=h, num_frames=num_frames,
                num_inference_steps=num_inference_steps, guidance_scale=guidance_scale,
                generator=gen, max_sequence_length=512,
            ).frames[0]
        except torch.cuda.OutOfMemoryError as e:
            gc.collect(); torch.cuda.empty_cache()
            return JSONResponse(status_code=507, content={
                "error": "CUDA OOM", "detail": str(e),
                "hint": "Lower resolution or duration."})
    return _frames_to_response(out, fps, final_seed, round(time.time() - t0, 1), w, h, I2V_MODEL_ID)

print("✅ FastAPI app defined. Endpoints: GET /health · POST /generate · POST /image")

In [ ]:
# ==========================================================================
#  CELL 6 — TINY sanity check (1 s x 9 frames, minimum possible).
#  If this OOMs, your GPU has <14 GB usable: lower width/height to 768x432.
#  Skip it if you are in a hurry; the API server works without it.
# ==========================================================================
import gc, torch
gc.collect(); torch.cuda.empty_cache()
print("Running 1 s x 9 frame sanity check (tiny workload) ...")
tiny = GenBody(
    prompt="a photorealistic city street at dusk, people walking, cinematic live-action footage",
    width=768, height=432, duration=1.0, fps=9,
    num_inference_steps=8, guidance_scale=5.0, seed=1,
)
try:
    r = generate(tiny)
    if hasattr(r, 'status_code') and r.status_code == 507:
        print("⚠️  Sanity OOM. Server still works — just use smaller prompts from your laptop.")
    else:
        print(f"✅ Sanity OK — MP4 size: {len(r.body):,} bytes,"
              f" seed={r.headers.get('X-Seed')},"
              f" elapsed={r.headers.get('X-Elapsed-S')} s")
except Exception as e:
    print("⚠️  Sanity raised", type(e).__name__ + ":", str(e)[:200])
    print("   (Server is still usable — this was just a warm-up.)")
finally:
    gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ==========================================================================
#  CELL 7 — Start the HTTP server and print a PUBLIC URL.
#  Copy the printed URL into your laptop's .env as:
#      COLAB_BASE_URL=https://xxxx-xxxx-xxxx-xxxx-xxxx.trycloudflare.com
#  (no trailing slash)
#
#  Kaggle has NO google.colab.proxy, so we use cloudflared first, then fall
#  back to localtunnel. Each method is time-boxed and failure-tolerant so a
#  blocked port NEVER crashes the kernel.
# ==========================================================================
import os, shutil, subprocess, threading, time
import uvicorn, nest_asyncio

nest_asyncio.apply()
PORT = 8000

config = uvicorn.Config(
    app, host="0.0.0.0", port=PORT,
    log_level="warning", lifespan="off", access_log=False,
)
server = uvicorn.Server(config)
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()

# Wait for the socket to bind before opening a tunnel.
time.sleep(3.0)

PUBLIC_URL = None
URL_METHOD = None
FAIL_LOG = []

# 1. Cloudflared (fast, reliable one-off binary). Works on most Kaggle nodes.
try:
    bin_cf = shutil.which("cloudflared")
    if bin_cf is None:
        import urllib.request
        url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
        print("Downloading cloudflared (~45 MB) ...")
        urllib.request.urlretrieve(url, "/tmp/cloudflared")
        os.chmod("/tmp/cloudflared", 0o755)
        bin_cf = "/tmp/cloudflared"
    logf = open("/tmp/cf.log", "w")
    cf_proc = subprocess.Popen(
        [bin_cf, "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
        stdout=logf, stderr=logf,
    )
    deadline = time.time() + 25
    while time.time() < deadline and PUBLIC_URL is None:
        time.sleep(1.0)
        try:
            txt = open("/tmp/cf.log").read()
            for token in txt.split():
                if token.startswith("https://") and ".trycloudflare.com" in token:
                    PUBLIC_URL, URL_METHOD = token.rstrip("/"), "cloudflared"
                    break
        except Exception:
            pass
    if PUBLIC_URL is None:
        cf_proc.terminate()
        FAIL_LOG.append(("cloudflared", "no URL within 25s"))
except Exception as e:
    FAIL_LOG.append(("cloudflared", str(e)))

# 2. LocalTunnel (fallback). npm must be present; timeout-boxed.
if PUBLIC_URL is None:
    try:
        import json, urllib.request
        print("Installing localtunnel ...")
        subprocess.check_call(["npm", "install", "-g", "localtunnel"],
                              stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        lt_proc = subprocess.Popen(["lt", "--port", str(PORT)],
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        deadline = time.time() + 15
        line = ""
        while time.time() < deadline and PUBLIC_URL is None:
            ch = lt_proc.stdout.read(1)
            if not ch:
                time.sleep(0.2); continue
            if ch == "\n":
                if "your url is:" in line.lower():
                    candidate = line.split(":", 1)[1].strip()
                    if candidate.startswith("http"):
                        PUBLIC_URL, URL_METHOD = candidate, "localtunnel"
                        break
                line = ""
            else:
                line += ch
        if PUBLIC_URL is None:
            lt_proc.terminate()
            FAIL_LOG.append(("localtunnel", "no URL within 15s"))
    except Exception as e:
        FAIL_LOG.append(("localtunnel", str(e)))

print()
print("=" * 72)
if PUBLIC_URL:
    print(f"✅  TUNNEL READY via {URL_METHOD}")
    print("\n   🔗 PUBLIC API URL — paste into your laptop's .env as COLAB_BASE_URL:")
    print()
    print("   ", PUBLIC_URL)
    print()
    print("   💡 Keep this Kaggle tab open. Idle sessions disconnect and the URL")
    print("      dies — re-run cell 7 and update COLAB_BASE_URL to the new URL.")
    print()
    print("   Health check:", PUBLIC_URL + "/health")
else:
    print("🛑  Both tunnel methods FAILED. Details:")
    for m, err in FAIL_LOG:
        print(f"   - {m:12s}: {err}")
    print()
    print("   Fixes: Settings → Internet ON, allow outbound HTTPS (Kaggle's free tier")
    print("   restricts some outbound ports). Or re-run this cell on a fresh session.")
print("=" * 72)

---

## Optional: quick test from inside Kaggle

Once a PUBLIC_URL is printed, test the endpoint end-to-end (text-to-video):

```python
import requests
r = requests.post(f"{PUBLIC_URL}/generate", json={
    "prompt": "photorealistic zombies walking down an abandoned city street at dusk, fog",
    "duration": 5.0, "width": 832, "height": 480, "fps": 16,
}, timeout=1800)
print("t2v:", r.status_code)
if r.status_code == 200:
    open("/kaggle/working/t2v_test.mp4", "wb").write(r.content)
    print("wrote /kaggle/working/t2v_test.mp4")
```

Then on your laptop add `COLAB_BASE_URL=<url>` to `.env` and use the studio
with the **colab** backend, or run:
`python -m src.main gen --backend colab "..."` — no code changes needed.